In [1]:
from datasets import load_dataset 

imdb = load_dataset("imdb")

c:\Users\sdsc\Desktop\proj\fine-tune-sdsc4070\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoTokenizer

MODEL = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

In [3]:
tokenized_imdb = imdb.map(tokenize, batched=True)
tokenized_imdb

Map: 100%|██████████| 50000/50000 [00:25<00:00, 1938.58 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [4]:
import numpy as np
from transformers import DataCollatorWithPadding
import evaluate

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")

def compute_scores(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [5]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label=id2label, label2id=label2id
)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 5654.61it/s]
GPT2ForSequenceClassification LOAD REPORT from: distilbert/distilgpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
import torch
from transformers import TrainerCallback

def gpu_memory_stats():
    if not torch.cuda.is_available():
        return {}

    device = torch.cuda.current_device()
    gib = 1024 ** 3
    total = torch.cuda.get_device_properties(device).total_memory / gib

    return {
        "gpu_mem_allocated_gb": round(torch.cuda.memory_allocated(device) / gib, 3),
        "gpu_mem_reserved_gb": round(torch.cuda.memory_reserved(device) / gib, 3),
        "gpu_mem_peak_allocated_gb": round(torch.cuda.max_memory_allocated(device) / gib, 3),
        "gpu_mem_total_gb": round(total, 3),
    }

class GPUMemoryCallback(TrainerCallback):
    def on_train_begin(self, _args, _state, _control, **_kwargs):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def on_log(self, _args, state, _control, logs=None, **_kwargs):
        if logs is None or not torch.cuda.is_available():
            return

        stats = gpu_memory_stats()
        logs.update(stats)
        print(
            f"step={state.global_step} "
            f"allocated={stats['gpu_mem_allocated_gb']:.3f} GB "
            f"reserved={stats['gpu_mem_reserved_gb']:.3f} GB "
            f"peak={stats['gpu_mem_peak_allocated_gb']:.3f} GB"
        )

gpu_memory_stats()

{'gpu_mem_allocated_gb': 0.0,
 'gpu_mem_reserved_gb': 0.0,
 'gpu_mem_peak_allocated_gb': 0.0,
 'gpu_mem_total_gb': 7.996}

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    skip_memory_metrics=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_scores,
    callbacks=[GPUMemoryCallback()],
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss,Accuracy,Mem Allocated Gb,Mem Reserved Gb,Mem Peak Allocated Gb,Mem Total Gb
1,0.223638,0.204326,0.923680,0.937000,10.002000,9.765000,7.996000


step=50 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=100 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=150 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=200 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=250 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=300 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=350 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=400 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=450 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=500 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.35s/it]


step=550 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=600 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=650 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=700 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=750 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=800 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=850 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=900 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=950 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB
step=1000 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.54s/it]


... (2 outputs left out)

step=1550 allocated=0.938 GB reserved=10.002 GB peak=9.765 GB


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.54s/it]


step=1563 allocated=0.937 GB reserved=10.002 GB peak=9.765 GB
step=1563 allocated=0.937 GB reserved=1.215 GB peak=9.765 GB


TrainOutput(global_step=1563, training_loss=0.2804194594985464, metrics={'train_runtime': 55541.8981, 'train_samples_per_second': 0.45, 'train_steps_per_second': 0.028, 'total_flos': 6532654694400000.0, 'train_loss': 0.2804194594985464, 'init_mem_cpu_alloc_delta': 1284247552, 'init_mem_gpu_alloc_delta': 328456192, 'init_mem_cpu_peaked_delta': 327655424, 'init_mem_gpu_peaked_delta': 0, 'train_mem_cpu_alloc_delta': -2407264256, 'train_mem_gpu_alloc_delta': 677988864, 'train_mem_cpu_peaked_delta': 7939874816, 'train_mem_gpu_peaked_delta': 9478367744, 'before_init_mem_cpu': 1768624128, 'before_init_mem_gpu': 0, 'epoch': 1.0, 'gpu_mem_allocated_gb': 0.937, 'gpu_mem_reserved_gb': 1.215, 'gpu_mem_peak_allocated_gb': 9.765, 'gpu_mem_total_gb': 7.996})